Do not Run All. Restart the kernel before the GPU run cells; each model runs in its own subprocess.
# Base versus fine-tuned Qwen3.5-4B comparison
This notebook runs the cached base model and the saved LoRA adapter under identical inference settings. It compares parsing, scorer metrics, per-sequence changes, and representative errors.


## 0. Setup and environment check
CPU-only setup checks the project paths and package versions. GPU cell: run manually applies only to the separate subprocess cells below. Check that the expected Python environment and cached model revision are visible.


In [ ]:
from pathlib import Path
import importlib.metadata as md
import json, os, platform, subprocess, sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs' / 'base.yaml').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from config import load_config
from collators import ThinkingOffProcessor
config = load_config(PROJECT_ROOT, profile='cuda_4060')
print({'python': sys.version, 'platform': platform.platform(), 'model': config.hf_model_name, 'revision': config.hf_model_revision})
for package in ('torch','transformers','unsloth','peft','bitsandbytes'):
    try: print(package, md.version(package))
    except md.PackageNotFoundError: print(package, 'not installed')


## 1. Settings
This CPU-only cell resolves the newest adapter, split, processor settings, and output directory. Check that the adapter path is the intended project adapter, not another Studio experiment.


In [ ]:
from datetime import datetime
weights_root = PROJECT_ROOT / 'weights'
adapter_candidates = [p.parent for p in weights_root.glob('*/adapter_config.json')]
if not adapter_candidates: raise FileNotFoundError('No adapter_config.json found under weights/')
adapter_path = max(adapter_candidates, key=lambda p: p.stat().st_mtime)
split_name = 'val'
limit = None
max_pixels = config.max_pixels
max_new_tokens = config.max_new_tokens
comparison_dir = PROJECT_ROOT / 'results' / f'compare_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
comparison_dir.mkdir(parents=True, exist_ok=True)
print({'adapter_path': str(adapter_path), 'split': split_name, 'max_pixels': max_pixels, 'max_new_tokens': max_new_tokens, 'comparison_dir': str(comparison_dir)})


## 2. Run BASE
GPU cell: run manually. This starts `src/infer.py` in a separate process with no adapter, so its CUDA memory is released before the fine-tuned run. Check the JSON summary, prompt, image-token count, parse failures, and truncation count.


In [ ]:
base_path = comparison_dir / 'base_predictions.jsonl'
base_command = [sys.executable, str(PROJECT_ROOT / 'src' / 'infer.py'), '--split', split_name, '--adapter', 'none', '--out', str(base_path), '--max-pixels', str(max_pixels), '--max-new-tokens', str(max_new_tokens), '--experiment', comparison_dir.name]
print(' '.join(repr(x) for x in base_command))
base_run = subprocess.run(base_command, cwd=PROJECT_ROOT, check=True, text=True, capture_output=False)


## 3. Run FINE-TUNED
GPU cell: run manually after the base subprocess finishes. It uses the identical CLI path and settings, adding only the selected adapter. Check that the first prompt and image-token count match the base run.


In [ ]:
finetuned_path = comparison_dir / 'finetuned_predictions.jsonl'
finetuned_command = [sys.executable, str(PROJECT_ROOT / 'src' / 'infer.py'), '--split', split_name, '--adapter', str(adapter_path), '--out', str(finetuned_path), '--max-pixels', str(max_pixels), '--max-new-tokens', str(max_new_tokens), '--experiment', comparison_dir.name]
print(' '.join(repr(x) for x in finetuned_command))
finetuned_run = subprocess.run(finetuned_command, cwd=PROJECT_ROOT, check=True, text=True, capture_output=False)


## 4. Optional train-subset comparison
GPU cell: run manually only if needed. This is off by default and runs both models on the first 16 training sequences to inspect a train-versus-validation gap.


In [ ]:
run_train_subset = False
if run_train_subset:
    for tag, adapter in [('base','none'), ('finetuned', str(adapter_path))]:
        out = comparison_dir / f'{tag}_train_subset_predictions.jsonl'
        command = [sys.executable, str(PROJECT_ROOT / 'src' / 'infer.py'), '--split', 'train_subset', '--adapter', adapter, '--out', str(out), '--max-pixels', str(max_pixels), '--max-new-tokens', str(max_new_tokens), '--experiment', comparison_dir.name]
        subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('train_subset disabled')


## 5. Score both prediction files
CPU-side orchestration calls the official scorer separately for base and fine-tuned outputs. `src/eval.py` writes filtered references in prediction order without editing the originals. Check that both score files exist and contain the scorer's reported keys.


In [ ]:
score_paths = {}
for tag, prediction_path in [('base', base_path), ('finetuned', finetuned_path)]:
    experiment = comparison_dir.name + '/' + tag
    subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / 'eval.py'), str(prediction_path), '--experiment', experiment], cwd=PROJECT_ROOT, check=True)
    score_paths[tag] = PROJECT_ROOT / 'results' / comparison_dir.name / tag / 'scores.json'
scores = {tag: json.loads(path.read_text(encoding='utf-8')) for tag, path in score_paths.items()}
base_meta = json.loads((base_path.parent / f'{base_path.stem}.metadata.json').read_text(encoding='utf-8'))
finetuned_meta = json.loads((finetuned_path.parent / f'{finetuned_path.stem}.metadata.json').read_text(encoding='utf-8'))
assert base_meta['first_prompt'] == finetuned_meta['first_prompt'], 'Base and fine-tuned prompts differ'
assert base_meta['first_image_tokens'] == finetuned_meta['first_image_tokens'], 'Base and fine-tuned image-token counts differ'
print({'scorer_keys': {tag: list(value.get('macro', value).keys()) for tag, value in scores.items()}, 'prompt_match': True, 'image_tokens_match': True})


## 6. Results table
This CPU-only cell combines scorer metrics with CLI metadata and saves `comparison.csv`. Check that both rows use the same revision, prompt settings, and max-pixels value.


In [ ]:
import csv
def metadata_for(path): return json.loads((path.parent / f'{path.stem}.metadata.json').read_text(encoding='utf-8'))
rows = []
for tag, prediction_path in [('base', base_path), ('finetuned', finetuned_path)]:
    macro = scores[tag].get('macro', scores[tag])
    meta = metadata_for(prediction_path)
    rows.append({'model': tag, **{key: macro.get(key) for key in macro}, 'parse_failures': meta.get('parse_failures'), 'truncated': meta.get('truncated'), 'mean_lines_per_page': meta.get('mean_lines_per_page'), 'seconds_per_sequence': meta.get('seconds_per_sequence'), 'peak_vram_gb': meta.get('peak_vram_gb'), 'adapter_sha256': meta.get('adapter_sha256'), 'base_revision': meta.get('base_revision'), 'prompt_name': 'PROMPT', 'max_pixels': max_pixels})
fieldnames = list(rows[0])
with (comparison_dir / 'comparison.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames); writer.writeheader(); writer.writerows(rows)
print(rows)


## 7. Paired per-sequence comparison
This CPU-only cell compares the scorer's per-sequence `joint_f1` values, falling back to `balanced_joint_f1` or `text_order_score` if needed. It reports improved/worse/tied counts, a fixed-seed bootstrap 95% interval, and a delta chart. Differences of only a point or two may be noise with 16 validation sequences.


In [ ]:
import random, statistics
def sequence_metric(item): return item.get('joint_f1', item.get('balanced_joint_f1', item.get('text_order_score', 0.0)))
base_seq, ft_seq = scores['base'].get('sequences', {}), scores['finetuned'].get('sequences', {})
paired = []
for sequence_id in sorted(set(base_seq) & set(ft_seq)):
    b, f = sequence_metric(base_seq[sequence_id]), sequence_metric(ft_seq[sequence_id])
    paired.append({'sequence_id': sequence_id, 'base': b, 'finetuned': f, 'delta': f - b})
deltas = [row['delta'] for row in paired]
rng = random.Random(42); boot = []
for _ in range(2000): boot.append(statistics.mean(rng.choice(deltas) for _ in deltas))
boot.sort(); interval = (boot[int(.025 * len(boot))], boot[int(.975 * len(boot)) - 1])
summary = {'improved': sum(x > 0 for x in deltas), 'worse': sum(x < 0 for x in deltas), 'tied': sum(x == 0 for x in deltas), 'mean_delta': statistics.mean(deltas), 'bootstrap_95': interval}
(comparison_dir / 'paired_comparison.json').write_text(json.dumps({'summary': summary, 'rows': paired}, indent=2), encoding='utf-8')
print(summary)
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12, 4)); plt.bar([x['sequence_id'] for x in paired], deltas); plt.axhline(0, color='black'); plt.xticks(rotation=90); plt.ylabel('fine-tuned minus base'); plt.tight_layout(); plt.savefig(comparison_dir / 'paired_deltas.png'); plt.show()
except ImportError:
    print('matplotlib unavailable; paired JSON was still saved')


## 8. Side-by-side viewer
CPU-side helper for inspecting one sequence. It prints reference, base, and fine-tuned lines in page order, then lists the three largest improvements, regressions, and invalid/empty cases.


In [ ]:
def read_jsonl(path): return {row['sequence_id']: row for row in (json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip())}
predictions = {'base': read_jsonl(base_path), 'finetuned': read_jsonl(finetuned_path)}
references = read_jsonl(PROJECT_ROOT / config.dataset_path / 'development' / 'labels.jsonl')
def show_sequence(sequence_id):
    for page in range(3):
        print(f'PAGE {page + 1}')
        print('REF:', references[sequence_id]['pages'][page])
        print('BASE:', predictions['base'][sequence_id]['pages'][page])
        print('FINE:', predictions['finetuned'][sequence_id]['pages'][page])
def empty_or_invalid(row): return not row.get('pages') or all(not page for page in row['pages'])
print('largest improvements', sorted(paired, key=lambda x: x['delta'], reverse=True)[:3])
print('largest regressions', sorted(paired, key=lambda x: x['delta'])[:3])
print('empty outputs', [sid for sid in predictions['base'] if empty_or_invalid(predictions['base'][sid]) or empty_or_invalid(predictions['finetuned'][sid])])
# Example: show_sequence(paired[0]['sequence_id'])


### 8.1 Visual inspection of one sequence
This CPU-only cell mirrors the video example using our three-page sequence format and saved predictions. It does not load a model; check the page images, reference lines, base output, and fine-tuned output side by side.


In [ ]:
from PIL import Image
from IPython.display import display
from prepare import load_examples
split_data = json.loads((PROJECT_ROOT / config.split_path).read_text(encoding='utf-8'))
val_examples_for_view = list(load_examples(PROJECT_ROOT / config.dataset_path, split_data['val']))
example_by_id = {example['sequence_id']: example for example in val_examples_for_view}
sequence_to_view = paired[0]['sequence_id'] if paired else next(iter(predictions['base']))
def inspect_sequence(sequence_id):
    example = example_by_id[sequence_id]
    print('SEQUENCE:', sequence_id)
    for page_number, image_path in enumerate(example['image_paths'], 1):
        print(f'PAGE {page_number}: {image_path}')
        image = Image.open(image_path).convert('RGB'); image.thumbnail((360, 360)); display(image)
        print('REFERENCE:', references[sequence_id]['pages'][page_number - 1])
        print('BASE:', predictions['base'][sequence_id]['pages'][page_number - 1])
        print('FINE-TUNED:', predictions['finetuned'][sequence_id]['pages'][page_number - 1])
inspect_sequence(sequence_to_view)


## 9. Rule-based error summary
These are counts, not causal diagnoses. Rules: exact reference text with a different speaker is a speaker error; absent reference text is missing; predicted text absent from references is possible invented text/SFX/intro text; differing matched-text order is a page-order difference.


In [ ]:
def error_counts(prediction_rows):
    counts = {'speaker_wrong': 0, 'missing_reference_lines': 0, 'possible_invented_lines': 0, 'page_order_different': 0}
    for sid, ref in references.items():
        pred = prediction_rows.get(sid, {'pages':[[],[],[]]})
        ref_lines = [line for page in ref['pages'] for line in page]
        pred_lines = [line for page in pred['pages'] for line in page]
        ref_text = [line.get('text','') for line in ref_lines]; pred_text = [line.get('text','') for line in pred_lines]
        for line in ref_lines:
            matches = [x for x in pred_lines if x.get('text') == line.get('text')]
            if matches and all(x.get('speaker') != line.get('speaker') for x in matches): counts['speaker_wrong'] += 1
            if line.get('text') not in pred_text: counts['missing_reference_lines'] += 1
        counts['possible_invented_lines'] += sum(text not in ref_text for text in pred_text)
        matched_ref = [text for text in ref_text if text in pred_text]; matched_pred = [text for text in pred_text if text in ref_text]
        if matched_ref != matched_pred: counts['page_order_different'] += 1
    return counts
error_summary = {'base': error_counts(predictions['base']), 'finetuned': error_counts(predictions['finetuned'])}
(comparison_dir / 'rule_based_error_summary.json').write_text(json.dumps(error_summary, indent=2), encoding='utf-8')
print(error_summary)


## 10. Reproducibility artifacts
This CPU-only cell records the exact subprocess commands, resolved settings, adapter hash, and package metadata next to the comparison outputs.


In [ ]:
commands = {'base': base_command, 'finetuned': finetuned_command}
(comparison_dir / 'commands.json').write_text(json.dumps(commands, indent=2), encoding='utf-8')
(comparison_dir / 'comparison_settings.json').write_text(json.dumps({'adapter_path': str(adapter_path), 'split': split_name, 'max_pixels': max_pixels, 'max_new_tokens': max_new_tokens, 'base_revision': config.hf_model_revision}, indent=2), encoding='utf-8')
print('Saved comparison artifacts under', comparison_dir)
